In [21]:
import requests
import pandas as pd
import os, pickle, time
from datetime import datetime, timedelta

In [23]:


API_KEY = "YOUR_API_KEY"  # get at polygon.io

def cached_download(ticker, cache_dir="stock_cache", retries=5, days=365):
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = f"{cache_dir}/{ticker}.pkl"

    # Return cached version if available
    if os.path.exists(cache_file):
        print(f"Loading {ticker} from cache")
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    # Default date range
    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days)).strftime("%Y-%m-%d")

    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start_date}/{end_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params)

            # Handle HTTP level errors
            if r.status_code == 403:
                raise ValueError("Invalid API key")
            if r.status_code == 429:
                raise ConnectionError("Rate limited")
            r.raise_for_status()

            data = r.json()

            # Handle Polygon response status
            if data.get("status") == "ERROR":
                raise ValueError(f"Polygon error: {data.get('error', 'Unknown error')}")
            if not data.get("results"):
                raise ValueError(f"No data returned for {ticker}")

            # Parse into a clean DataFrame
            df = pd.DataFrame(data["results"])
            df = df.rename(columns={
                "t": "timestamp",
                "o": "open",
                "h": "high",
                "l": "low",
                "c": "close",
                "v": "volume",
                "vw": "vwap"
            })
            df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
            df = df.set_index("date")
            df = df[["open", "high", "low", "close", "volume", "vwap"]]

            # Cache and return
            with open(cache_file, "wb") as f:
                pickle.dump(df, f)
            print(f"Downloaded and cached {ticker}")
            return df

        except ValueError as e:
            # Don't retry on bad ticker or API key
            if "Invalid API key" in str(e) or "Polygon error" in str(e):
                raise
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)

        except (ConnectionError, requests.exceptions.RequestException) as e:
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} — network error: {e}. Waiting {wait}s...")
            time.sleep(wait)

    raise Exception(f"Failed to download {ticker} after {retries} retries")

In [24]:
# Usage
df = cached_download("AAPL", days=365)

Downloaded and cached AAPL
                       open      high     low   close      volume      vwap
date                                                                       
2025-03-14 04:00:00  211.25  213.9500  209.58  213.49  60107582.0  212.4529
2025-03-17 04:00:00  213.31  215.2200  209.97  214.00  48073426.0  213.2536
2025-03-18 04:00:00  214.16  215.1500  211.49  212.69  42432426.0  213.0584
2025-03-19 04:00:00  214.22  218.7600  213.75  215.24  54385391.0  215.5599
2025-03-20 04:00:00  213.99  217.4899  212.22  214.10  48862947.0  214.3888


In [25]:
df

,open,high,low,close,volume,vwap
date,,,,,,
2025-03-14 04:00:00,211.250,213.9500,209.5800,213.49,6.010758e+07,212.4529
2025-03-17 04:00:00,213.310,215.2200,209.9700,214.00,4.807343e+07,213.2536
2025-03-18 04:00:00,214.160,215.1500,211.4900,212.69,4.243243e+07,213.0584
2025-03-19 04:00:00,214.220,218.7600,213.7500,215.24,5.438539e+07,215.5599
2025-03-20 04:00:00,213.990,217.4899,212.2200,214.10,4.886295e+07,214.3888
...,...,...,...,...,...,...
2026-03-09 04:00:00,255.690,261.1500,253.6805,259.88,3.821853e+07,258.0266
2026-03-10 04:00:00,257.645,262.4800,256.9500,260.83,3.059077e+07,260.7116
2026-03-11 04:00:00,261.090,262.1300,259.5500,260.81,2.621893e+07,260.6671
